# NYC Building Forecasts via BBL

**Goal:** Retrieve building-level probabilistic forecasts for New York City properties using BBL (Borough-Block-Lot) identifiers from the NYC PLUTO dataset.

## BBL Format
NYC uses a Borough-Block-Lot system:
- **Borough:** 1=Manhattan, 2=Bronx, 3=Brooklyn, 4=Queens, 5=Staten Island
- **Block:** 5-digit zero-padded block number
- **Lot:** 4-digit zero-padded lot number
- **Combined:** `BBBBBLLLL` — e.g. `1005390013` = Manhattan, Block 539, Lot 13

Find BBL numbers at [nycpaymentportal.nyc.gov](https://nycpaymentportal.nyc.gov) or via the [NYC PLUTO API](https://data.cityofnewyork.us/City-Government/Primary-Land-Use-Tax-Lot-Output-PLUTO-/64uk-42ks).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/homecastr/homecastr-cookbook/blob/main/examples/getting_started/04_nyc_buildings.ipynb)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from homecastr import HomecastrClient

client = HomecastrClient(os.getenv("HOMECASTR_API_KEY", "hc_demo_public_readonly"))

In [ ]:
# Manhattan building — confirmed live data
BBL = "1005390013"  # Manhattan, Block 539, Lot 13

fan_df = client.forecast.by_parcel.retrieve_fan_chart(BBL)

print(f"BBL:           {fan_df.attrs['acct']}")
print(f"Current value: ${fan_df.attrs['current_value']:,.0f}")
print(f"Origin year:   {fan_df.attrs['origin_year']}")
print()
print(fan_df[['year', 'horizon_months', 'p10', 'p50', 'p90']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.fill_between(fan_df["year"], fan_df["p10"] / 1e6, fan_df["p90"] / 1e6,
                alpha=0.15, color="crimson", label="P10–P90")
ax.fill_between(fan_df["year"], fan_df["p25"] / 1e6, fan_df["p75"] / 1e6,
                alpha=0.3, color="crimson", label="P25–P75")
ax.plot(fan_df["year"], fan_df["p50"] / 1e6, color="crimson", lw=2, label="P50 (median)")
ax.axhline(fan_df.attrs["current_value"] / 1e6, color="gray", ls="--", lw=1, label="Current value")

ax.set_title(f"NYC Building Forecast — BBL {BBL}\n(Manhattan, Block 539, Lot 13)")
ax.set_xlabel("Year")
ax.set_ylabel("Value ($M)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.1f}M"))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compare multiple NYC buildings
NYC_BBLS = {
    "Manhattan Block 539 Lot 13": "1005390013",
    "Manhattan Block 539 Lot 14": "1005390014",
    "Manhattan Block 539 Lot 16": "1005390016",
}

rows = []
for label, bbl in NYC_BBLS.items():
    result = client.forecast.by_parcel.retrieve(bbl)
    fan = result.get("fan_chart", [])
    h60 = next((f for f in fan if f.get("horizon_months") == 60), {})
    rows.append({
        "label": label,
        "bbl": result.get("acct", bbl),
        "current_value": result.get("current_value"),
        "p50_5yr": h60.get("p50"),
        "appreciation_5yr": (
            round((h60["p50"] / result["current_value"] - 1) * 100, 1)
            if h60.get("p50") and result.get("current_value") else None
        ),
    })

df = pd.DataFrame(rows)
df["current_value"] = df["current_value"].apply(lambda x: f"${x:,.0f}" if x else "—")
df["p50_5yr"] = df["p50_5yr"].apply(lambda x: f"${x:,.0f}" if x else "—")
print(df.to_string(index=False))